### Lab 4.2: Batching and Regularization

In this lab you will learn how to set up a dataset to be processed in batches, rather than processing the entire dataset in each training iteration, and explore neural network regularization.

In [132]:
import numpy as np
import torch
from matplotlib import pyplot as plt

We will use the `diabetes` datasets from scikit-learn:

*Ten baseline variables, age, sex, body mass index, average blood pressure, and six blood serum measurements were obtained for each of n = 442 diabetes patients, as well as the response of interest, a quantitative measure of disease progression one year after baseline.*

This is a regression dataset, so we will use mean squared error as the loss function.

In [133]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X = diabetes['data']
y = diabetes['target'][:,None]
X.shape, X.dtype, y.shape, y.dtype

((442, 10), dtype('float64'), (442, 1), dtype('float64'))

In [134]:
X.min(axis=0), X.max(axis=0)

(array([-0.10722563, -0.04464164, -0.0902753 , -0.1123988 , -0.12678067,
        -0.11561307, -0.10230705, -0.0763945 , -0.12609712, -0.13776723]),
 array([0.11072668, 0.05068012, 0.17055523, 0.13204362, 0.15391371,
        0.19878799, 0.18117906, 0.18523444, 0.13359728, 0.13561183]))

In [135]:
y.min(), y.max()

(np.float64(25.0), np.float64(346.0))

To make the learning algorithm work more smoothly, we we will subtract the mean of each feature.

Here `np.mean` calculates a mean, and `axis=0` tells NumPy to calculate the mean over the rows (calculate the mean of each column).

In [136]:
X -= np.mean(X,axis=0)
X /= np.std(X,axis=0)
y -= np.mean(y,axis=0)
y /= np.std(y,axis=0)

Now we will convert our `X` and `y` arrays to torch Tensors.

In [137]:
X = torch.tensor(X).float()
y = torch.tensor(y).float()

### Exercises

1. Divide the data into train and test splits (`sklearn.model_selection.train_test_split`)
2. Create a neural network for this dataset.
3. Use `TensorDataset` and `DataLoader` to batch the dataset during training.  
4. Use `weight_decay` parameter to `optim.SGD` to introduce L2 regularization during training. Evaluate the effect of regularization on test set accuracy.

*Note: make sure to report the BEST test accuracy seen during training, not the last!*

In [138]:
from sklearn.model_selection import train_test_split

# 1. Train/test split
train_X, test_X, train_y, test_y = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert to tensors
train_X_t = torch.tensor(train_X, dtype=torch.float32)
train_y_t = torch.tensor(train_y, dtype=torch.float32)
test_X_t = torch.tensor(test_X, dtype=torch.float32)
test_y_t = torch.tensor(test_y, dtype=torch.float32)

n_features = train_X_t.shape[1]

/tmp/ipykernel_7905/1137603219.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_X_t = torch.tensor(train_X, dtype=torch.float32)
/tmp/ipykernel_7905/1137603219.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_y_t = torch.tensor(train_y, dtype=torch.float32)
/tmp/ipykernel_7905/1137603219.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_X_t = torch.tensor(test_X, dtype=torch.float32)
/tmp/ipykernel_7905/1137603219.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach()

In [139]:
import torch.nn as nn
# 2. Neural network
class Net(nn.Module):
    def __init__(self, n_features, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)

In [140]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
# 3. TensorDataset + DataLoader
train_ds = TensorDataset(train_X_t, train_y_t)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

def train_model(weight_decay, epochs=200):
    model = Net(n_features)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=weight_decay)

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        test_mse = criterion(model(test_X_t), test_y_t).item()
    return test_mse

In [141]:
# 4. Compare weight_decay values
for wd in [0.0, 1e-4, 1e-3, 1e-2, 1e-1]:
    mse = train_model(weight_decay=wd)
    print(f"weight_decay={wd:<8} test MSE={mse:.4f}")

weight_decay=0.0      test MSE=0.4522
weight_decay=0.0001   test MSE=0.5013
weight_decay=0.001    test MSE=0.4327
weight_decay=0.01     test MSE=0.4439
weight_decay=0.1      test MSE=0.4594
